# Festa em Cena — análise de vendas Halloween

**Sprint 1:** configuração, validação do arquivo e ingestão com `csv.DictReader`.

Cada **linha** do CSV é um pedido de um único produto (pode haver várias unidades em `quantidade`). Os horários em `data_venda` são locais (São Paulo), sem fuso explícito.

## Significado de colunas (amostra)

| Coluna | Significado |
|--------|-------------|
| `pedido_id` | Identificador do pedido |
| `data_venda` | Data/hora do registro da venda |
| `canal_venda` | Mercado, Loja física ou Web |
| `status_pedido` | Situação do pedido (ex.: Concluído, Cancelado) |

In [ ]:
from pathlib import Path

from functions import (
    DEFAULT_CSV_NAME,
    EXPECTED_COLUMNS,
    default_csv_path,
    read_sales_csv,
    validate_csv_file,
)

csv_path = default_csv_path()
print(f"Arquivo esperado: {csv_path}")
print(f"Existe no disco: {csv_path.is_file()}")

## Validação estrutural do CSV

Antes de analisar os dados, verificamos existência, UTF-8, cabeçalho e colunas obrigatórias.

In [ ]:
validation = validate_csv_file(csv_path)
validation.summary()

In [ ]:
if not validation.ok:
    raise SystemExit("Corrija o arquivo CSV antes de continuar.")

print(f"Colunas esperadas ({len(EXPECTED_COLUMNS)}): {', '.join(EXPECTED_COLUMNS)}")
print(f"Total de linhas de dados: {validation.row_count}")

## Leitura com `csv.DictReader`

Todos os campos são mantidos como **texto**; conversões numéricas e de data ficam para sprints posteriores.

In [ ]:
registros = read_sales_csv(csv_path)
len(registros), list(registros[0].keys())

In [ ]:
print("Três registros de exemplo:\n")
for i, row in enumerate(registros[:3], start=1):
    print(f"--- Registro {i} (pedido_id={row['pedido_id']}) ---")
    for chave in EXPECTED_COLUMNS:
        print(f"  {chave}: {row[chave]}")
    print()

## Sprint 2 — padronização de strings

Limpeza de `categoria` com `.strip()`, `.lower()` e `re.sub()` (pontuação removida; letras e espaços preservados). Canais reconhecidos passam a **Mercado**, **Loja física** e **Web**. O arquivo CSV no disco não é alterado.

In [ ]:
from functions import transform_string_fields

registros_limpos, stats_strings = transform_string_fields(registros)
stats_strings.summary()

In [ ]:
def exemplos_antes_depois(
    original: list[dict[str, str]],
    transformado: list[dict[str, str]],
    campo: str,
    limite: int = 3,
) -> list[tuple[str, str]]:
    pares: list[tuple[str, str]] = []
    for antes, depois in zip(original, transformado):
        if antes[campo] != depois[campo]:
            pares.append((antes[campo], depois[campo]))
        if len(pares) >= limite:
            break
    return pares

print("Categoria (até 3 exemplos):")
for antes, depois in exemplos_antes_depois(registros, registros_limpos, "categoria"):
    print(f"  {antes!r} -> {depois!r}")

print("\nCanal de venda (até 3 exemplos):")
for antes, depois in exemplos_antes_depois(registros, registros_limpos, "canal_venda"):
    print(f"  {antes!r} -> {depois!r}")